# Работа с данными

In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)


In [3]:
path = 'data/BTCUSDT_spot_1d_monthly_2020-01_2025-12.csv'

df = pd.read_csv(path)
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'], utc=True, errors='coerce')
df = df.sort_values('datetime_utc').reset_index(drop=True)
df.shape

(2192, 10)

In [4]:
df.head(3)

,datetime_utc,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote
0,2020-01-01 00:00:00+00:00,7195.24,7255.0,7175.15,7200.85,16792.388165,1.212145e+08,194010,8946.955535,6.459779e+07
1,2020-01-02 00:00:00+00:00,7200.77,7212.5,6924.74,6965.71,31951.483932,2.259823e+08,302667,15141.611340,1.070608e+08
2,2020-01-03 00:00:00+00:00,6965.49,7405.0,6871.04,7344.96,68428.500451,4.950986e+08,519854,35595.496273,2.577131e+08


In [5]:
df.tail(3)

,datetime_utc,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote
2189,2025-12-29 00:00:00+00:00,87952.71,90406.08,86806.50,87237.13,19894.98575,1.762012e+09,4793589,10079.10102,8.922982e+08
2190,2025-12-30 00:00:00+00:00,87237.13,89400.00,86845.66,88485.49,13105.91001,1.155016e+09,3542176,6561.56089,5.783034e+08
2191,2025-12-31 00:00:00+00:00,88485.50,89200.00,87250.00,87648.22,11558.62047,1.019747e+09,2897396,5432.03408,4.791050e+08


In [6]:
checks = {}

checks['rows'] = len(df)
checks['datetime_nulls'] = int(df['datetime_utc'].isna().sum())
checks['datetime_duplicates'] = int(df['datetime_utc'].duplicated().sum())
checks['is_sorted'] = bool(df['datetime_utc'].is_monotonic_increasing)

dt = df['datetime_utc'].dropna()
if len(dt) > 1:
    gaps = dt.diff().dt.total_seconds().div(86400)
    gap_counts = gaps.value_counts(dropna=True).sort_index()
    checks['min_gap_days'] = float(gaps.min())
    checks['max_gap_days'] = float(gaps.max())
else:
    gap_counts = pd.Series(dtype='int64')

checks, gap_counts.head(10), gap_counts.tail(10)


({'rows': 2192,
  'datetime_nulls': 0,
  'datetime_duplicates': 0,
  'is_sorted': True,
  'min_gap_days': 1.0,
  'max_gap_days': 1.0},
 datetime_utc
 1.0    2191
 Name: count, dtype: int64,
 datetime_utc
 1.0    2191
 Name: count, dtype: int64)

In [7]:
num_cols = [
    'open', 'high', 'low', 'close', 'volume',
    'quote_volume', 'taker_buy_base', 'taker_buy_quote'
]
int_cols = ['trades']

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

df['trades'] = pd.to_numeric(df['trades'], errors='coerce').astype('Int64')

before = len(df)
df = df.dropna(subset=['datetime_utc', 'open', 'high', 'low', 'close']).copy()
after = len(df)

bad_high = (df['high'] < df[['open', 'close']].max(axis=1)).sum()
bad_low = (df['low'] > df[['open', 'close']].min(axis=1)).sum()
nonpos_close = (df['close'] <= 0).sum()

{
    'dropped_rows_due_to_nans': before - after,
    'bad_high_count': int(bad_high),
    'bad_low_count': int(bad_low),
    'nonpositive_close_count': int(nonpos_close)
}


{'dropped_rows_due_to_nans': 0,
 'bad_high_count': 0,
 'bad_low_count': 0,
 'nonpositive_close_count': 0}

Доходность, лог-доходность, дневной ценовой размах, True Range, Average True Range, Годовая волатильность на основе доходностей

In [8]:
df = df.set_index('datetime_utc').sort_index()

df['ret_1d'] = df['close'].pct_change()
df['logret_1d'] = np.log(df['close']).diff()

df['range'] = (df['high'] - df['low']) / df['close'].shift(1)

prev_close = df['close'].shift(1)
tr = pd.concat(
    [
        (df['high'] - df['low']),
        (df['high'] - prev_close).abs(),
        (df['low'] - prev_close).abs()
    ],
    axis=1
).max(axis=1)

df['true_range'] = tr
df['atr_14'] = df['true_range'].rolling(14, min_periods=14).mean()

df['vol_30'] = df['ret_1d'].rolling(30, min_periods=30).std() * np.sqrt(365)

df[['open', 'high', 'low', 'close', 'volume', 'ret_1d', 'logret_1d', 'atr_14', 'vol_30']].tail(10)


,open,high,low,close,volume,ret_1d,logret_1d,atr_14,vol_30
datetime_utc,,,,,,,,,
2025-12-22 00:00:00+00:00,88658.87,90588.23,87900.00,88620.79,14673.21970,-0.000429,-0.000429,3318.988571,0.408714
2025-12-23 00:00:00+00:00,88620.79,88940.00,86601.90,87486.00,13910.32904,-0.012805,-0.012888,3122.496429,0.403088
2025-12-24 00:00:00+00:00,87486.00,88049.89,86420.00,87669.45,9140.84320,0.002097,0.002095,3030.856429,0.398759
2025-12-25 00:00:00+00:00,87669.44,88592.74,86934.72,87225.27,7096.58235,-0.005067,-0.005079,2842.545714,0.397401
2025-12-26 00:00:00+00:00,87225.27,89567.75,86655.08,87369.56,18344.61505,0.001654,0.001653,2816.736429,0.376322
2025-12-27 00:00:00+00:00,87369.56,87984.00,87253.05,87877.01,4469.55156,0.005808,0.005791,2806.935714,0.375285
2025-12-28 00:00:00+00:00,87877.00,88088.75,87435.00,87952.71,4446.29285,0.000861,0.000861,2646.843571,0.375095
2025-12-29 00:00:00+00:00,87952.71,90406.08,86806.50,87237.13,19894.98575,-0.008136,-0.008169,2553.527857,0.375941
2025-12-30 00:00:00+00:00,87237.13,89400.00,86845.66,88485.49,13105.91001,0.014310,0.014209,2528.125000,0.379487


In [9]:
df[['open', 'high', 'low', 'close', 'volume', 'ret_1d', 'logret_1d', 'atr_14', 'vol_30']].head(31)

,open,high,low,close,volume,ret_1d,logret_1d,atr_14,vol_30
datetime_utc,,,,,,,,,
2020-01-01 00:00:00+00:00,7195.24,7255.00,7175.15,7200.85,16792.388165,NaN,NaN,NaN,NaN
2020-01-02 00:00:00+00:00,7200.77,7212.50,6924.74,6965.71,31951.483932,-0.032654,-0.033200,NaN,NaN
2020-01-03 00:00:00+00:00,6965.49,7405.00,6871.04,7344.96,68428.500451,0.054445,0.053015,NaN,NaN
2020-01-04 00:00:00+00:00,7345.00,7404.00,7272.21,7354.11,29987.974977,0.001246,0.001245,NaN,NaN
2020-01-05 00:00:00+00:00,7354.19,7495.00,7318.00,7358.75,38331.085604,0.000631,0.000631,NaN,NaN
2020-01-06 00:00:00+00:00,7357.64,7795.34,7346.76,7758.00,54635.695316,0.054255,0.052834,NaN,NaN
2020-01-07 00:00:00+00:00,7758.90,8207.68,7723.71,8145.28,91171.684661,0.049920,0.048714,NaN,NaN
2020-01-08 00:00:00+00:00,8145.92,8455.00,7870.00,8055.98,112622.642640,-0.010963,-0.011024,NaN,NaN
2020-01-09 00:00:00+00:00,8054.72,8055.96,7750.00,7817.76,64239.519830,-0.029571,-0.030017,NaN,NaN


In [10]:
keep_cols = [
    'open', 'high', 'low', 'close', 'volume',
    'quote_volume', 'trades', 'taker_buy_base', 'taker_buy_quote',
    'ret_1d', 'logret_1d', 'range', 'true_range', 'atr_14', 'vol_30'
]

df_bt = df[keep_cols].copy()

required = ['ret_1d', 'atr_14', 'vol_30']
df_bt_clean = df_bt.dropna(subset=required).copy()

df_bt_clean.shape

(2162, 15)

In [11]:
out_path = 'data/btc_spot_2020-2025.csv'
df_bt_clean.to_csv(out_path, index=True)
out_path

'data/btc_spot_2020-2025.csv'

In [12]:
df_bt_clean

,open,high,low,close,volume,quote_volume,trades,taker_buy_base,taker_buy_quote,ret_1d,logret_1d,range,true_range,atr_14,vol_30
datetime_utc,,,,,,,,,,,,,,,
2020-01-31 00:00:00+00:00,9511.52,9530.22,9210.01,9352.89,45552.022352,4.257790e+08,428547,22482.470899,2.101503e+08,-0.016852,-0.016996,0.033660,320.21,342.576429,0.584214
2020-02-01 00:00:00+00:00,9351.71,9464.53,9281.00,9384.61,28578.067354,2.681377e+08,334972,14679.126305,1.377641e+08,0.003391,0.003386,0.019623,183.53,342.650000,0.564925
2020-02-02 00:00:00+00:00,9384.41,9477.03,9120.00,9331.51,45690.912540,4.280641e+08,446558,22588.406060,2.118082e+08,-0.005658,-0.005674,0.038044,357.03,315.796429,0.544472
2020-02-03 00:00:00+00:00,9331.59,9618.79,9234.00,9292.24,50892.133451,4.763800e+08,492124,24781.397696,2.320659e+08,-0.004208,-0.004217,0.041236,384.79,327.159286,0.545704
2020-02-04 00:00:00+00:00,9291.35,9350.00,9093.01,9197.02,53308.175266,4.911361e+08,528930,26103.432307,2.405715e+08,-0.010247,-0.010300,0.027656,256.99,324.015714,0.548916
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-12-27 00:00:00+00:00,87369.56,87984.00,87253.05,87877.01,4469.551560,3.912742e+08,957536,2065.724050,1.808605e+08,0.005808,0.005791,0.008366,730.95,2806.935714,0.375285
2025-12-28 00:00:00+00:00,87877.00,88088.75,87435.00,87952.71,4446.292850,3.903273e+08,1241426,2230.752160,1.958421e+08,0.000861,0.000861,0.007439,653.75,2646.843571,0.375095
2025-12-29 00:00:00+00:00,87952.71,90406.08,86806.50,87237.13,19894.985750,1.762012e+09,4793589,10079.101020,8.922982e+08,-0.008136,-0.008169,0.040926,3599.58,2553.527857,0.375941
